In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import sys, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
from brainomni.model import BrainOmni
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print('Device:', device)

In [ ]:
# Frozen BrainOmni + NN head baseline.
# Backbone is ENTIRELY frozen; only the NN head trains. Same head + same 100-epoch
# nested-validation protocol as finetune_neuralnet, so this fills the missing
# "frozen + NN head" cell for an apples-to-apples comparison.
CKPT_PATH = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection/tiny'
SAVE_DIR  = '../data/preprocessed'
OUT_DIR   = '../results/figures'
os.makedirs(OUT_DIR, exist_ok=True)

with open(os.path.join(CKPT_PATH, 'model_cfg.json')) as f:
    model_config = json.load(f)
base_checkpoint = torch.load(os.path.join(CKPT_PATH, 'BrainOmni.pt'), map_location='cpu', weights_only=True)

proj_out1 = np.load(os.path.join(SAVE_DIR, 'proj_out1.npy'))
proj_out2 = np.load(os.path.join(SAVE_DIR, 'proj_out2.npy'))
labels    = np.load(os.path.join(SAVE_DIR, 'labels.npy'))
N = len(labels)

FROZEN_LR_CV = 0.831   # frozen + Logistic Regression baseline
HANDCRAFT_CV = 0.872   # handcrafted + LR baseline

def make_split(labels, holdout, seed=123):
    idx = np.arange(len(labels))
    tr, tmp = train_test_split(idx, test_size=2*holdout, stratify=labels, random_state=seed)
    va, te  = train_test_split(tmp, test_size=0.5, stratify=labels[tmp], random_state=seed)
    return tr, va, te
SPLITS = {'80/10/10': make_split(labels, 0.10), '70/15/15': make_split(labels, 0.15)}
print('N =', N)

In [ ]:
def fresh_frozen_model():
    m = BrainOmni(**model_config)
    m.load_state_dict(base_checkpoint, strict=False)
    for p in m.parameters():
        p.requires_grad = False
    return m.to(device).eval()

@torch.no_grad()
def compute_frozen_features(model):
    """Every subject through ALL frozen blocks (blocks[:-1]) -> per-subject feature,
    identical to the fine-tune head's input but with the backbone frozen.
    Averages the two 30s segments."""
    feats = []
    for i in range(N):
        x1 = torch.tensor(proj_out1[i]).unsqueeze(0).to(device)
        x2 = torch.tensor(proj_out2[i]).unsqueeze(0).to(device)
        for block in list(model.blocks)[:-1]:
            x1 = block(x1); x2 = block(x2)
        x1 = F.normalize(x1, p=2.0, dim=-1, eps=1e-6).mean(dim=2)
        x2 = F.normalize(x2, p=2.0, dim=-1, eps=1e-6).mean(dim=2)
        feats.append(((x1 + x2) / 2).flatten(1).squeeze(0).cpu().numpy())
    return np.array(feats, dtype=np.float32)

_model = fresh_frozen_model()
FEATS = compute_frozen_features(_model)
print('Frozen features:', FEATS.shape)

In [ ]:
class FeatDataset(Dataset):
    def __init__(self, idx): self.idx = idx
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        return torch.tensor(FEATS[j]), torch.tensor(labels[j], dtype=torch.long)

def make_head():
    # SAME architecture as finetune_neuralnet's head
    return nn.Sequential(nn.Linear(FEATS.shape[1], 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2))

def make_warmup_cosine(optimizer, total_steps, warmup_ratio=0.1, warmup_min=0.1, cos_min=0.1):
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    def lr_lambda(step):
        if step < warmup_steps:
            return warmup_min + (1 - warmup_min) * step / warmup_steps
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return cos_min + (1 - cos_min) * 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

HEAD_LR = 1e-4

def train_head(tr_idx, va_idx, max_epochs=250, patience=10**9, use_val=True, monitor_idx=None):
    tl_loader = DataLoader(FeatDataset(tr_idx), batch_size=8, shuffle=True)
    vl_loader = DataLoader(FeatDataset(va_idx), batch_size=8) if use_val else None
    mn_loader = DataLoader(FeatDataset(monitor_idx), batch_size=8) if monitor_idx is not None else None
    head = make_head().to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=0.05, betas=(0.9, 0.99), eps=1e-6)
    sched = make_warmup_cosine(opt, max_epochs * len(tl_loader))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    def ev(loader):
        head.eval(); p, t, ls = [], [], 0.0
        with torch.no_grad():
            for x, y in loader:
                out = head(x.to(device)); ls += crit(out, y.to(device)).item()
                p.extend(out.argmax(1).cpu().numpy()); t.extend(y.numpy())
        return accuracy_score(t, p), ls / len(loader)
    best_vl, best_ep, best_w = float('inf'), 0, None
    ta, vacc, tl, vl, mona, monl = [], [], [], [], [], []
    for ep in range(max_epochs):
        head.train(); preds, trues, el = [], [], 0.0
        for x, y in tl_loader:
            opt.zero_grad(); out = head(x.to(device)); loss = crit(out, y.to(device)); loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step(); sched.step()
            el += loss.item(); preds.extend(out.argmax(1).cpu().numpy()); trues.extend(y.numpy())
        tra, trl = accuracy_score(trues, preds), el / len(tl_loader)
        va_, vll = ev(vl_loader) if vl_loader is not None else (0.0, trl)
        if mn_loader is not None:
            ma_, mll = ev(mn_loader); mona.append(ma_); monl.append(mll)
        ta.append(tra); vacc.append(va_); tl.append(trl); vl.append(vll)
        if vll < best_vl - 1e-4:
            best_vl, best_ep = vll, ep + 1
            best_w = {k: v.clone() for k, v in head.state_dict().items()}
    if best_w is not None:
        head.load_state_dict(best_w)
    hist = {'best_ep': best_ep, 'train_accs': ta, 'val_accs': vacc, 'train_losses': tl, 'val_losses': vl}
    if mn_loader is not None:
        hist['mon_accs'] = mona; hist['mon_losses'] = monl
    return head, hist

@torch.no_grad()
def head_predict(head, idx):
    head.eval(); yt, yp, ypr = [], [], []
    for i in idx:
        out = head(torch.tensor(FEATS[i]).unsqueeze(0).to(device))
        ypr.append(torch.softmax(out, 1)[0, 1].item()); yp.append(out.argmax(1).item()); yt.append(labels[i])
    return np.array(yt), np.array(yp), np.array(ypr)
print('Head + training helpers ready.')

In [ ]:
def run_split(split):
    tr, va, te = split
    head, curves = train_head(tr, va, max_epochs=250)
    yt, yp, _ = head_predict(head, te)
    curves['test_acc'] = accuracy_score(yt, yp)
    return curves

def run_cv(cv_epochs=250, n_splits=5, seed=123, inner_val=0.15):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full(N, -1, int); pred = np.zeros(N, int); prob = np.zeros(N); folds, fh = [], []
    for f, (tr, te) in enumerate(skf.split(np.arange(N), labels)):
        itr, iva = train_test_split(tr, test_size=inner_val, stratify=labels[tr], random_state=seed)
        head, hist = train_head(itr, iva, max_epochs=cv_epochs, patience=cv_epochs, use_val=True, monitor_idx=te)
        fh.append(hist)
        yt, yp, ypr = head_predict(head, te)
        pred[te], prob[te], oof[te] = yp, ypr, labels[te]; folds.append(accuracy_score(labels[te], yp))
        print(f'  fold {f+1}: best_ep {hist["best_ep"]} | acc {folds[-1]:.3f}')
    return {'mean': float(np.mean(folds)), 'folds': folds, 'y_true': oof, 'pred': pred, 'prob': prob, 'fold_hist': fh}
print('Eval helpers ready.')

In [ ]:
# Two fixed splits
split_res = {}
for name, sp in SPLITS.items():
    split_res[name] = run_split(sp)
    print(f'{name}: best epoch {split_res[name]["best_ep"]} | test {split_res[name]["test_acc"]:.3f}')

In [ ]:
# 5-fold CV (nested validation)
print('=== Frozen + NN head | 5-fold CV ===')
cv = run_cv()
print(f'Frozen + NN head CV mean: {cv["mean"]:.3f}  (baseline frozen+LR = {FROZEN_LR_CV})')

In [ ]:
# 5-fold CV learning curve: mean and +/-1 std band across folds.
# Blue=train, orange=inner val (selection), green=held-out fold (diagnostic).
# Check that the curves flatten well before the 250 cap = converged.
hs = cv['fold_hist']
L  = min(len(h['train_losses']) for h in hs)
ep = np.arange(1, L + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
def band(ax, key, color, label):
    M = np.array([h[key][:L] for h in hs]); m, s = M.mean(0), M.std(0)
    ax.plot(ep, m, color=color, label=label); ax.fill_between(ep, m - s, m + s, color=color, alpha=0.2)
band(axes[0], 'train_accs', 'steelblue',  'Train')
band(axes[0], 'val_accs',   'darkorange', 'Inner val (selection)')
band(axes[0], 'mon_accs',   'seagreen',   'Held-out fold (diagnostic)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].set_ylim(0.4, 1.0)
axes[0].set_title(f'CV acc {cv["mean"]:.3f}'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
band(axes[1], 'train_losses', 'steelblue',  'Train')
band(axes[1], 'val_losses',   'darkorange', 'Inner val (selection)')
band(axes[1], 'mon_losses',   'seagreen',   'Held-out fold (diagnostic)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
fig.suptitle('Frozen BrainOmni-tiny + NN head: 5-fold CV learning curve (mean and +/-1 std)', fontsize=12)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'frozen_nn_curves_cv.png'), dpi=150); plt.show()
print('Saved frozen_nn_curves_cv.png')

In [ ]:
# Comparison bar chart
labels_x = ['80/10/10', '70/15/15', '5-fold CV']
vals = [split_res['80/10/10']['test_acc'], split_res['70/15/15']['test_acc'], cv['mean']]
errs = [0, 0, float(np.std(cv['folds']))]
colors = ['#4E79A7', '#F28E2B', '#B07AA1']
fig, ax = plt.subplots(figsize=(7, 5))
b = ax.bar(labels_x, vals, yerr=errs, capsize=5, color=colors)
ax.bar_label(b, fmt='%.3f')
ax.axhline(FROZEN_LR_CV, color='green', ls='--', lw=1.3, label=f'Frozen + LR (CV, {FROZEN_LR_CV})')
ax.axhline(HANDCRAFT_CV, color='red', ls=':', lw=1.3, label=f'Handcrafted + LR (CV, {HANDCRAFT_CV})')
ax.set_ylabel('Accuracy'); ax.set_ylim(0.5, 1.0)
ax.set_title('Frozen BrainOmni-tiny + NN head (backbone frozen)')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'frozen_nn_comparison.png'), dpi=150); plt.show()
print('Saved frozen_nn_comparison.png')

In [ ]:
torch.save({'split_res': split_res, 'cv': cv, 'feats_shape': list(FEATS.shape)},
           os.path.join(SAVE_DIR, 'frozen_nn.pt'))
print('Saved frozen_nn.pt')